In [1]:
from google.colab import files
uploaded = files.upload()

Saving promos.csv to promos.csv
Saving stakes_delivery.csv to stakes_delivery.csv


In [7]:
#imports
import pandas as pd
import io

In [4]:
DATA_DIR = "."

In [6]:
promos = pd.read_csv(f"{DATA_DIR}/promos.csv")
print(promos.shape)

(120, 6)


In [8]:
def load_stakes(path):

    rows = []

    with open(path, encoding="utf-8-sig") as f:

        header = f.readline().strip().split(",")

        for line in f:
            line = line.rstrip("\r\n")
            if not line.strip():
                continue
            parts = line.split(",")
            if len(parts) < 5:
                rows.append(("malformed", line))
                continue
            site_id = parts[0]
            site_name = ",".join(parts[1:-3])
            game, date, stakes = parts[-3], parts[-2], parts[-1]
            rows.append((site_id, site_name, game, date, stakes))
    return rows, header

rows, header = load_stakes("stakes_delivery.csv")
print(header, len(rows))

['site_id', 'site_name', 'game', 'date', 'stakes_eur'] 81692


In [9]:
df = pd.DataFrame(rows, columns=header)
bad = df[df["site_id"] == "malformed"]
stakes = df[df["site_id"] != "malformed"].copy()

stakes["site_id"] = pd.to_numeric(stakes["site_id"])
stakes["date"] = pd.to_datetime(stakes["date"], errors="coerce")
stakes["stakes_eur"] = pd.to_numeric(stakes["stakes_eur"], errors="coerce")

In [10]:
print(len(bad))
print(stakes.shape)
print(stakes.isna().sum())
print(stakes.duplicated(["site_id", "game", "date"]).sum())

0
(81692, 5)
site_id       0
site_name     0
game          0
date          0
stakes_eur    0
dtype: int64
51


In [11]:
key = ["site_id", "game", "date"]
dups = stakes[stakes.duplicated(key, keep=False)].sort_values(key)

print(dups.head(20))
print("exact duplicates:", stakes.duplicated().sum())
print("same key but different value:", dups.drop_duplicates().duplicated(key).sum())

       site_id        site_name           game       date  stakes_eur
72301     1004  Casino Delta IT   Copper Crown 2026-09-10     1725.51
81653     1004  Casino Delta IT   Copper Crown 2026-09-10     1725.51
72748     1004  Casino Delta IT   Copper Crown 2026-09-11     1658.04
81666     1004  Casino Delta IT   Copper Crown 2026-09-11     1658.04
73195     1004  Casino Delta IT   Copper Crown 2026-09-12     2049.97
81679     1004  Casino Delta IT   Copper Crown 2026-09-12     2049.97
72302     1004  Casino Delta IT   Crystal Reel 2026-09-10     1253.26
81654     1004  Casino Delta IT   Crystal Reel 2026-09-10     1253.26
72749     1004  Casino Delta IT   Crystal Reel 2026-09-11     1350.16
81667     1004  Casino Delta IT   Crystal Reel 2026-09-11     1350.16
73196     1004  Casino Delta IT   Crystal Reel 2026-09-12     1694.29
81680     1004  Casino Delta IT   Crystal Reel 2026-09-12     1694.29
72303     1004  Casino Delta IT  Diamond Drift 2026-09-10      561.90
81655     1004  Casi

In [12]:
print(stakes["date"].min(), stakes["date"].max())
print(stakes["stakes_eur"].describe())
print("negative:", (stakes["stakes_eur"] < 0).sum())
print("zero:", (stakes["stakes_eur"] == 0).sum())
print(stakes.groupby(["site_id", "site_name"]).size())
print(stakes["game"].value_counts().sort_index())

2026-04-01 00:00:00 2026-09-30 00:00:00
count    81692.000000
mean      1257.415507
std        828.469119
min       -132.160000
25%        686.340000
50%       1048.080000
75%       1561.367500
max       8703.610000
Name: stakes_eur, dtype: float64
negative: 12
zero: 0
site_id  site_name        
1001     Casino Alpha AT      2013
1002     Casino Beta, DE      2562
1003     Casino Gamma HR      1853
1004     Casino Delta IT      2419
1005     Casino Epsilon RO    1647
1006     Casino Zeta AT       1647
1007     Bet Eta NL           1647
1008     Slots Theta ES       1647
1009     Casino Iota CH       2013
1010     Bet Kappa SI         1647
1011     Casino Lambda LV     2379
1012     Casino Mu RS         1648
1013     Slots Nu BG          2013
1014     Bet Xi DE            2562
1015     Bet Omicron NL       2562
1016     Casino Pi IT         2196
1017     Slots Rho ES         1647
1018     Casino Sigma HR      2196
1019     Bet Tau CH           1830
1020     Casino Upsilon AT    2562
102

In [13]:
stakes = stakes.drop_duplicates().reset_index(drop=True)

In [14]:
key = ["site_id", "game", "date"]
conf = stakes[stakes.duplicated(key, keep=False) & ~stakes.duplicated(keep=False)].sort_values(key)
print(conf)

       site_id         site_name            game       date  stakes_eur
47765     1004   Casino Delta IT   Lucky Lantern 2026-07-17     2680.74
47766     1004   Casino Delta IT   Lucky Lantern 2026-07-17      -36.59
63012     1012      Casino Mu RS    Copper Crown 2026-08-20     1004.56
63013     1012      Casino Mu RS    Copper Crown 2026-08-20      -84.30
33708     1023        Bet Psi IT      Ruby Spins 2026-06-15      492.89
33709     1023        Bet Psi IT      Ruby Spins 2026-06-15      -74.87
70744     1024      Bet Omega SI  Thunder Hooves 2026-09-06      659.17
70745     1024      Bet Omega SI  Thunder Hooves 2026-09-06      -73.59
7893      1028       Bet Dune LV     Sunken Gold 2026-04-18      651.58
7894      1028       Bet Dune LV     Sunken Gold 2026-04-18      -41.97
59648     1029   Slots Falcon ES    Tiger Lagoon 2026-08-12     1261.77
59649     1029   Slots Falcon ES    Tiger Lagoon 2026-08-12      -50.17
52992     1033  Slots Juniper RO    Copper Crown 2026-07-28     

In [15]:
neg = stakes[stakes["stakes_eur"] < 0]
print(len(neg))

12


In [16]:
stakes = (stakes.groupby(["site_id", "site_name", "game", "date"], as_index=False)["stakes_eur"].sum())

print(stakes.shape)
print(stakes.duplicated(["site_id", "game", "date"]).sum())

(81641, 5)
0


In [17]:
print(stakes["date"].min(), stakes["date"].max())
print(stakes["stakes_eur"].describe())
print("neg:", (stakes["stakes_eur"] < 0).sum())
print("zeros:", (stakes["stakes_eur"] == 0).sum())

2026-04-01 00:00:00 2026-09-30 00:00:00
count    81641.000000
mean      1257.541904
std        828.476789
min        118.740000
25%        686.480000
50%       1047.940000
75%       1561.350000
max       8703.610000
Name: stakes_eur, dtype: float64
neg: 0
zeros: 0


In [18]:
print(stakes.groupby("site_id")["site_name"].nunique().loc[lambda s: s > 1])

Series([], Name: site_name, dtype: int64)


In [19]:
mb = stakes[stakes["game"] == "Midnight Bells"]
print(mb.groupby(["site_id", "site_name"])["date"].agg(["min", "max", "count"]))

                               min        max  count
site_id site_name                                   
1003    Casino Gamma HR 2026-09-08 2026-09-30     23


In [20]:
span = stakes.groupby(["site_id", "game"])["date"].agg(["min", "max", "count"])
span["expected"] = (span["max"] - span["min"]).dt.days + 1
span["missing"] = span["expected"] - span["count"]
print(span[span["missing"] > 0])
print("site/game pairs:", len(span))

Empty DataFrame
Columns: [min, max, count, expected, missing]
Index: []
site/game pairs: 447


# promo analysis

In [21]:
print(promos.shape)
print(promos.dtypes)
print(promos.to_string())

(120, 6)
promo_id      object
site_id        int64
site_name     object
games         object
start_date    object
end_date      object
dtype: object
    promo_id  site_id          site_name                                     games  start_date    end_date
0        P01     1001    Casino Alpha AT                     Golden Fox;Ruby Spins  2026-08-17  2026-08-23
1        P02     1002    Casino Beta, DE                            Pharaoh's Gate  2026-09-07  2026-09-13
2        P03     1003    Casino Gamma HR                             Lucky Lantern  2026-08-24  2026-08-30
3        P04     1004    Casino Delta IT                                       ALL  2026-09-14  2026-09-20
4        P05     1001    Casino Alpha AT                              Crystal Reel  2026-09-28  2026-09-22
5        P06     1004    Casino Delta IT                             Lucky Lantern  2026-08-10  2026-08-16
6        P07     1003    Casino Gamma HR                            Midnight Bells  2026-09-09  2026-0

In [22]:
promos["start_date"] = pd.to_datetime(promos["start_date"], errors="coerce")
promos["end_date"] = pd.to_datetime(promos["end_date"], errors="coerce")
promos["n_days"] = (promos["end_date"] - promos["start_date"]).dt.days + 1

print(promos.isna().sum())
print("duplicated ids:", promos["promo_id"].duplicated().sum())
print(promos[promos["end_date"] < promos["start_date"]])

promo_id      0
site_id       0
site_name     0
games         0
start_date    0
end_date      0
n_days        0
dtype: int64
duplicated ids: 0
  promo_id  site_id        site_name         games start_date   end_date  \
4      P05     1001  Casino Alpha AT  Crystal Reel 2026-09-28 2026-09-22   

   n_days  
4      -5  


In [23]:
names = stakes.drop_duplicates("site_id").set_index("site_id")["site_name"]
print(promos[promos["site_id"].map(names) != promos["site_name"]])

Empty DataFrame
Columns: [promo_id, site_id, site_name, games, start_date, end_date, n_days]
Index: []


In [24]:
site_games = stakes.groupby("site_id")["game"].apply(set).to_dict()
missing = []

for _, r in promos.iterrows():
    if r["games"].strip().upper() == "ALL":
        continue
    for g in r["games"].split(";"):
        if g.strip() not in site_games.get(r["site_id"], set()):
            missing.append((r["promo_id"], r["site_id"], g))

print(missing)

[]


In [25]:
def compute_promo(r, stakes):
    s, e = r["start_date"], r["end_date"]
    out = {"promo_id": r["promo_id"], "before": None, "during": None, "uplift": None, "status": "ok"}

    if e < s:
        out["status"] = "invalid dates"
        return out

    n = (e - s).days + 1
    b_start, b_end = s - pd.Timedelta(days=n), s - pd.Timedelta(days=1)

    d = stakes[stakes["site_id"] == r["site_id"]]
    if r["games"].strip().upper() == "ALL":
        games = sorted(d["game"].unique())
    else:
        games = [g.strip() for g in r["games"].split(";")]
    d = d[d["game"].isin(games)]

    before = d[(d["date"] >= b_start) & (d["date"] <= b_end)]
    during = d[(d["date"] >= s) & (d["date"] <= e)]
    out["before"] = round(before["stakes_eur"].sum(), 2)
    out["during"] = round(during["stakes_eur"].sum(), 2)

    if b_start < pd.Timestamp("2026-04-01"):
        out["status"] = "before window starts before data"
    elif e > pd.Timestamp("2026-09-30"):
        out["status"] = "promo ends after data"
    elif len(before) < n * len(games):
        out["status"] = "incomplete before data"
    elif len(during) < n * len(games):
        out["status"] = "incomplete during data"
    elif out["before"] == 0:
        out["status"] = "before is zero"

    if out["status"] == "ok":
        out["uplift"] = round((out["during"] - out["before"]) / out["before"] * 100, 1)
    return out

results = pd.DataFrame([compute_promo(r, stakes) for _, r in promos.iterrows()])
print(results["status"].value_counts())
print(results.head())

status
ok                        118
invalid dates               1
incomplete before data      1
Name: count, dtype: int64
  promo_id     before     during  uplift         status
0      P01   10330.27   13645.02    32.1             ok
1      P02    3494.32    4154.51    18.9             ok
2      P03   11949.19   16990.55    42.2             ok
3      P04  120403.41  136922.69    13.7             ok
4      P05        NaN        NaN     NaN  invalid dates


In [26]:
print(results[results["status"] != "ok"])

  promo_id  before   during  uplift                  status
4      P05     NaN      NaN     NaN           invalid dates
6      P07  936.99  9063.78     NaN  incomplete before data


In [27]:
def games_of(r):
    if r["games"].strip().upper() == "ALL":
        return set(stakes.loc[stakes["site_id"] == r["site_id"], "game"])
    return {g.strip() for g in r["games"].split(";")}

valid = promos[promos["end_date"] >= promos["start_date"]]
pairs = []
for _, a in valid.iterrows():
    n = (a["end_date"] - a["start_date"]).days + 1
    w_start = a["start_date"] - pd.Timedelta(days=n)
    for _, b in valid.iterrows():
        if a["promo_id"] == b["promo_id"] or a["site_id"] != b["site_id"]:
            continue
        if not (games_of(a) & games_of(b)):
            continue
        if b["start_date"] <= a["end_date"] and b["end_date"] >= w_start:
            pairs.append((a["promo_id"], b["promo_id"], a["site_id"], sorted(games_of(a) & games_of(b))[:3]))

print(len(pairs))
for p in pairs:
    print(p)

0


In [28]:
ok = results[results["status"] == "ok"]

print(ok["uplift"].describe())
print(ok.sort_values("uplift").head(5))
print(ok.sort_values("uplift").tail(5))

count    118.000000
mean      24.481356
std       21.928045
min      -20.900000
25%        8.525000
50%       23.300000
75%       40.750000
max       67.300000
Name: uplift, dtype: float64
    promo_id    before    during  uplift status
58       P59  35341.87  27941.47   -20.9     ok
82       P83  12497.30  10138.46   -18.9     ok
87       P88  22157.77  18253.53   -17.6     ok
31       P32  12144.89  10567.47   -13.0     ok
106     P107  17213.42  15308.55   -11.1     ok
    promo_id    before    during  uplift status
89       P90   9122.29  14694.06    61.1     ok
65       P66  20607.94  33525.38    62.7     ok
64       P65   9095.10  14862.83    63.4     ok
47       P48   4176.30   6950.00    66.4     ok
118     P119   3472.48   5807.83    67.3     ok


In [29]:
g = stakes.groupby(["site_id", "game"])["stakes_eur"]
stakes["ratio"] = stakes["stakes_eur"] / g.transform("median")

print(stakes["ratio"].describe())
print(stakes.sort_values("ratio", ascending=False).head(15))
print(stakes.sort_values("ratio").head(10))

count    81641.000000
mean         1.027685
std          0.150482
min          0.610213
25%          0.918627
50%          1.000000
75%          1.122556
max          2.232048
Name: ratio, dtype: float64
       site_id          site_name           game       date  stakes_eur  \
60064     1030  Casino Glacier RS    Ember Wheel 2026-04-18     2695.22   
49161     1024       Bet Omega SI   Velvet Vault 2026-07-04     3277.34   
12434     1007         Bet Eta NL  Diamond Drift 2026-08-29     3112.04   
78413     1039     Bet Prairie CH  Lucky Lantern 2026-06-06     2172.23   
16010     1009     Casino Iota CH  Diamond Drift 2026-06-06     5275.71   
45508     1023         Bet Psi IT   Copper Crown 2026-07-11     2319.15   
66799     1033   Slots Juniper RO    Jade Temple 2026-09-12     6094.47   
66806     1033   Slots Juniper RO    Jade Temple 2026-09-19     6086.59   
78793     1039     Bet Prairie CH  Rapid Rooster 2026-06-20     2199.51   
45874     1023         Bet Psi IT     Frost Pe

In [30]:
stakes = stakes.drop(columns="ratio")

In [31]:
out = results[["promo_id", "before", "during", "uplift"]]
out.to_csv("results.csv", index=False)
results.to_json("results.json", orient="records")
print(out.shape)
print(out.head())

(120, 4)
  promo_id     before     during  uplift
0      P01   10330.27   13645.02    32.1
1      P02    3494.32    4154.51    18.9
2      P03   11949.19   16990.55    42.2
3      P04  120403.41  136922.69    13.7
4      P05        NaN        NaN     NaN


In [32]:
placebo = promos.copy()
placebo["start_date"] = promos["start_date"] - pd.to_timedelta(promos["n_days"], unit="D")
placebo["end_date"] = promos["start_date"] - pd.Timedelta(days=1)
pl = pd.DataFrame([compute_promo(r, stakes) for _, r in placebo.iterrows()])
print(pl["status"].value_counts())
print(pl["uplift"].describe())

status
ok                                  112
before window starts before data      6
invalid dates                         1
incomplete before data                1
Name: count, dtype: int64
count    112.000000
mean       0.583036
std        6.378933
min      -19.400000
25%       -2.000000
50%        0.700000
75%        3.100000
max       28.600000
Name: uplift, dtype: float64


In [33]:
print((promos["n_days"] % 7 == 0).value_counts())
stakes["dow"] = stakes["date"].dt.dayofweek
print((stakes.groupby("dow")["stakes_eur"].mean() / stakes["stakes_eur"].mean()).round(2))
stakes = stakes.drop(columns="dow")

n_days
True     86
False    34
Name: count, dtype: int64
dow
0    0.91
1    0.89
2    0.91
3    0.95
4    1.02
5    1.19
6    1.13
Name: stakes_eur, dtype: float64


In [34]:
print(out.shape)
print(out["uplift"].isna().sum())
print(out.dtypes)

(120, 4)
2
promo_id     object
before      float64
during      float64
uplift      float64
dtype: object


In [35]:
out.to_csv("results.csv", index=False)
results.to_json("results.json", orient="records")